<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not available in Colab Secrets.")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [24]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created.")

DuckDB connection created.


In [25]:
try:
    con.execute("DROP SECRET hf_token")
except:
    pass

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured for DuckDB.")

Hugging Face token configured for DuckDB.


In [26]:
test_query = """
SELECT COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

test_result = con.sql(test_query).df()

display(test_result)

,row_count
0,9841378


## 1. My rule and its reason codes

### Signal checks

**Signal 1 — GSC clicks**

The March 2026 data shows a strong separation between very-low-click content and content with more clicks. 138,050 content items have 0–2 clicks, while 38,688 have more than 2 clicks.

**Verdict: CONFIRMED**

Lower observed GSC clicks are used as a directional signal for higher review priority.

**Signal 2 — GSC average position**

The position buckets show progressively worse search position: median position increases from 3.45 in the first bucket to 34.98 in the fourth bucket.

**Verdict: CONFIRMED**

Worse observed average position is used as a directional signal for higher review priority.

Baseline rule: I use one simple action score based only on observed March 2026 signals. Lower GSC clicks receive a higher score, and worse GSC average position receives a higher score. Higher scores indicate stronger search-performance concerns and receive higher review priority.

The baseline uses one reason code and one action label:

LOW_SEARCH_PERFORMANCE — low observed clicks and/or poor observed search position.
REVIEW_REFRESH — prioritize the content item for human review.

This is a directional decision-support baseline and does not use future labels or future-window data.

### Signal 1 — GSC clicks

I expect content with lower observed GSC clicks to have a stronger need for review because fewer clicks indicate weaker search response.

Verdict: CONFIRMED

### Signal 2 — GSC average position

I expect content with worse average search position to have a stronger need for review because lower visibility can indicate an opportunity for improvement.

Verdict: CONFIRMED

These are directional baseline signals only. They use March 2026 observed data and do not use the future April label.

In [27]:
# ML-07 Section 1
# Two signal checks using March 2026 observed data only

march_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
    SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

march_df = con.sql(march_query).df()

print("March content-client rows:", len(march_df))
display(march_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March content-client rows: 176738


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,ga4_total_engagement_sec
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,NaN,NaN
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,8.240351,NaN,NaN
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,6.015432,NaN,NaN
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,5.956862,NaN,NaN
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,12.977513,NaN,NaN


In [28]:
import pandas as pd

march_df["click_bucket"] = pd.qcut(
    march_df["gsc_clicks"],
    q=4,
    duplicates="drop"
)

click_bucket = (
    march_df.groupby("click_bucket", observed=True)
    .agg(
        n=("gsc_clicks", "size"),
        mean_clicks=("gsc_clicks", "mean"),
        median_clicks=("gsc_clicks", "median")
    )
    .reset_index()
)

display(click_bucket)

,click_bucket,n,mean_clicks,median_clicks
0,"(-0.001, 2.0]",138050,0.286635,0.0
1,"(2.0, 5668.0]",38688,20.219758,9.0


In [29]:
march_df["position_bucket"] = pd.qcut(
    march_df["gsc_avg_position"],
    q=4,
    duplicates="drop"
)

position_bucket = (
    march_df.groupby("position_bucket", observed=True)
    .agg(
        n=("gsc_avg_position", "size"),
        mean_position=("gsc_avg_position", "mean"),
        median_position=("gsc_avg_position", "median")
    )
    .reset_index()
)

display(position_bucket)

,position_bucket,n,mean_position,median_position
0,"(-0.001, 5.002]",44185,3.235551,3.449750
1,"(5.002, 8.505]",44184,6.684165,6.631590
2,"(8.505, 20.369]",44184,13.114878,12.451883
3,"(20.369, 309.0]",44185,40.962239,34.984821


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [30]:
# ML-07 Section 2
# Build baseline action score using March 2026 observed data only

import os
import pandas as pd

# Start from March observed data
score_df = march_df.copy()

# ---------------------------------------------------------
# 1. Click signal
# Lower clicks = greater concern
# ---------------------------------------------------------

score_df["click_score"] = 0

score_df.loc[
    score_df["gsc_clicks"] <= 2,
    "click_score"
] = 2

score_df.loc[
    (score_df["gsc_clicks"] > 2) &
    (score_df["gsc_clicks"] <= 10),
    "click_score"
] = 1


# ---------------------------------------------------------
# 2. Search position signal
# Higher position number = weaker ranking
# ---------------------------------------------------------

score_df["position_score"] = 0

score_df.loc[
    score_df["gsc_avg_position"] > 5,
    "position_score"
] = 1

score_df.loc[
    score_df["gsc_avg_position"] > 10,
    "position_score"
] = 2

score_df.loc[
    score_df["gsc_avg_position"] > 20,
    "position_score"
] = 3


# ---------------------------------------------------------
# 3. Combined baseline score
# Maximum score = 5
# ---------------------------------------------------------

score_df["score"] = (
    score_df["click_score"] +
    score_df["position_score"]
)


# ---------------------------------------------------------
# 4. One reason code and one action label
# ---------------------------------------------------------

score_df["reason_code"] = "LOW_SEARCH_PERFORMANCE"
score_df["action"] = "REVIEW_REFRESH"


# ---------------------------------------------------------
# 5. Rank from highest score to lowest score
# ---------------------------------------------------------

score_df = score_df.sort_values(
    by=[
        "score",
        "gsc_clicks",
        "gsc_avg_position"
    ],
    ascending=[
        False,
        True,
        False
    ]
).reset_index(drop=True)

score_df["rank"] = range(
    1,
    len(score_df) + 1
)


# ---------------------------------------------------------
# 6. Create final ranked queue
# ---------------------------------------------------------

baseline_queue = score_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()


# ---------------------------------------------------------
# 7. Write CSV
# ---------------------------------------------------------

os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# 8. Verification
# ---------------------------------------------------------

print("Baseline queue created successfully.")
print("Rows:", len(baseline_queue))
print("Output:", output_path)

print("\nTop 10:")
display(baseline_queue.head(10))

print("\nScore distribution:")
print(
    baseline_queue["score"]
    .value_counts()
    .sort_index()
)

print("\nAction distribution:")
print(
    baseline_queue["action"]
    .value_counts()
)

print("\nReason distribution:")
print(
    baseline_queue["reason_code"]
    .value_counts()
)

print("\nCSV exists:", os.path.exists(output_path))

Baseline queue created successfully.
Rows: 176738
Output: work/outputs/baseline_action_score.csv

Top 10:


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
1,2,client_3ffa76342f366962,content_06589faf15cc8488,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
2,3,client_3ffa76342f366962,content_36cc2bda86ee726a,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
3,4,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
4,5,client_3ffa76342f366962,content_efce4eda2b012964,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
5,6,client_3ffa76342f366962,content_0cec599cfeab8b7f,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_61b375eafb1d4c27,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
7,8,client_f623b01661d4bfe4,content_fc468c5940d16ea3,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
8,9,client_3ffa76342f366962,content_3758dd311e8033f7,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
9,10,client_3ffa76342f366962,content_d1b44ca865290810,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH



Score distribution:
score
0     7795
1    12391
2    38691
3    48545
4    29159
5    40157
Name: count, dtype: int64

Action distribution:
action
REVIEW_REFRESH    176738
Name: count, dtype: int64

Reason distribution:
reason_code
LOW_SEARCH_PERFORMANCE    176738
Name: count, dtype: int64

CSV exists: True


In [31]:
# Verify the generated CSV

check_df = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)

print("CSV loaded successfully.")
print("CSV rows:", len(check_df))
print("CSV columns:", list(check_df.columns))
print("Duplicate rows:", check_df.duplicated().sum())
print("Missing values:")
print(check_df.isnull().sum())

CSV loaded successfully.
CSV rows: 176738
CSV columns: ['rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action']
Duplicate rows: 0
Missing values:
rank               0
client_hash_id     0
content_hash_id    0
score              0
reason_code        0
action             0
dtype: int64


## 3. Top-20 review

The top 20 rows are reviewed as decision-support recommendations. The action is `REVIEW_REFRESH` and the reason code is `LOW_SEARCH_PERFORMANCE`.

Confidence is directional because the baseline uses only two observed March signals. A recommendation could be wrong if the content is intentionally low-volume, newly published, highly seasonal, serving a narrow audience, or has valid business reasons for its current search performance.

For each item, I record what would make the recommendation wrong rather than treating the score as a definitive decision.

In [32]:
# ML-07 Section 3
# Generate the top-20 review table

top20 = score_df.head(20).copy()

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()

top20_review["confidence_note"] = (
    "Directional baseline based on observed clicks and position."
)

top20_review["what_would_make_it_wrong"] = (
    "Seasonality, intentional low-volume content, new content, "
    "or valid business context could make this recommendation wrong."
)

display(top20_review)

,rank,client_hash_id,content_hash_id,score,reason_code,action,gsc_clicks,gsc_avg_position,confidence_note,what_would_make_it_wrong
0,1,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,309.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
1,2,client_3ffa76342f366962,content_06589faf15cc8488,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,297.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
2,3,client_3ffa76342f366962,content_36cc2bda86ee726a,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,292.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
3,4,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,286.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
4,5,client_3ffa76342f366962,content_efce4eda2b012964,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,283.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
5,6,client_3ffa76342f366962,content_0cec599cfeab8b7f,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,280.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
6,7,client_23a62021009f63c4,content_61b375eafb1d4c27,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,271.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
7,8,client_f623b01661d4bfe4,content_fc468c5940d16ea3,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,262.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
8,9,client_3ffa76342f366962,content_3758dd311e8033f7,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,260.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."
9,10,client_3ffa76342f366962,content_d1b44ca865290810,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH,0.0,258.0,Directional baseline based on observed clicks ...,"Seasonality, intentional low-volume content, n..."


In [33]:
print("Top-20 rows:", len(top20_review))
print("Expected rows: 20")

assert len(top20_review) == 20

print("TOP-20 CHECK: PASS")

Top-20 rows: 20
Expected rows: 20
TOP-20 CHECK: PASS


## 4. Weak picks + leakage check

Some high-scoring items may be weak picks because low clicks or poor position do not automatically mean that a page should be refreshed. Content may be intentionally narrow, seasonal, newly published, or serving a specialized audience.

The baseline uses only March 2026 observed performance fields. It does not use the April 2026 future label, future clicks, future performance, or product flags. Therefore the baseline is designed to avoid future-window and label leakage.

In [34]:
# ML-07 Section 4
# Weak-pick and leakage checks

print("Potential weak-pick review:")

display(
    top20_review[
        [
            "rank",
            "score",
            "gsc_clicks",
            "gsc_avg_position",
            "action",
            "what_would_make_it_wrong"
        ]
    ].head(10)
)


# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

forbidden_columns = [
    "is_declining_label",
    "april_clicks",
    "future_clicks",
    "future_performance"
]

used_columns = set(score_df.columns)

found_leakage = [
    col
    for col in forbidden_columns
    if col in used_columns
]

print("\nLeakage columns found:", found_leakage)

if len(found_leakage) == 0:
    print("LEAKAGE CHECK: PASS")
else:
    print("LEAKAGE CHECK: REVIEW REQUIRED")


# ---------------------------------------------------------
# Confirm only March observed signals drive the score
# ---------------------------------------------------------

print("\nScoring inputs:")
print("gsc_clicks")
print("gsc_avg_position")

print("\nFuture-label column present:",
      "is_declining_label" in score_df.columns)

print(
    "SECTION 4 CHECK:",
    "PASS" if len(found_leakage) == 0 else "FAIL"
)

Potential weak-pick review:


,rank,score,gsc_clicks,gsc_avg_position,action,what_would_make_it_wrong
0,1,5,0.0,309.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
1,2,5,0.0,297.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
2,3,5,0.0,292.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
3,4,5,0.0,286.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
4,5,5,0.0,283.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
5,6,5,0.0,280.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
6,7,5,0.0,271.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
7,8,5,0.0,262.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
8,9,5,0.0,260.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."
9,10,5,0.0,258.0,REVIEW_REFRESH,"Seasonality, intentional low-volume content, n..."



Leakage columns found: []
LEAKAGE CHECK: PASS

Scoring inputs:
gsc_clicks
gsc_avg_position

Future-label column present: False
SECTION 4 CHECK: PASS


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal checks are completed with visible bucket tables and `n`
- [x] One baseline rule with a score, reason code, and action label is implemented
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`
- [x] The Top-20 review and weak-pick review are completed
- [x] The leakage check passes and no future-window or label-derived inputs are used
- [x] Committed to my repo under `work/notebooks/` and ready to submit the repo URL on the card. Done.